# all

## Libraries

In [4]:
from common import *

DATA = Path("../data")
engine = get_engine()

## Creating SQL server layers

In [3]:
with engine.connect() as conn:
    print("Database:", conn.execute(text("SELECT DB_NAME()")).scalar())
    print("Login:", conn.execute(text("SELECT SUSER_NAME()")).scalar())
    print(pd.read_sql(
        "SELECT name FROM sys.schemas "
        "WHERE name IN ('bronze', 'silver', 'gold')",
        conn
    ))

Database: Datacube
Login: datacube_analytics
     name
0  bronze
1    gold
2  silver


## Updating the schemas in layers with datasets

In [36]:
for table, file in [("airlines", "Flight/airlines.csv"), ("airports", "Flight/airports.csv")]:
    pd.read_csv(DATA / file).to_sql(table, engine, schema="bronze", if_exists="replace", index=False)
    print(table, "yükləndi")

airlines yükləndi
airports yükləndi


In [4]:
def load_parquet(table, path, columns=None, dtype=None):
    """Stream a parquet file into bronze.<table>; the first batch replaces the table."""
    for i, batch in enumerate(pq.ParquetFile(path).iter_batches(batch_size=100_000, columns=columns)):
        batch.to_pandas().to_sql(
            table, engine, schema="bronze",
            if_exists="replace" if i == 0 else "append",
            index=False, chunksize=5_000, dtype=dtype,
        )
        print(table, "batch", i, flush=True)

In [5]:
FLIGHT_COLS = [
    "YEAR", "MONTH", "DAY", "DAY_OF_WEEK", "AIRLINE",
    "ORIGIN_AIRPORT", "DESTINATION_AIRPORT", "DISTANCE",
    "SCHEDULED_DEPARTURE", "SCHEDULED_ARRIVAL",
    "DEPARTURE_DELAY", "ARRIVAL_DELAY",
    "AIR_SYSTEM_DELAY", "SECURITY_DELAY", "AIRLINE_DELAY",
    "LATE_AIRCRAFT_DELAY", "WEATHER_DELAY",
    "CANCELLED", "CANCELLATION_REASON", "DIVERTED",
    "TAXI_OUT", "TAXI_IN", "AIR_TIME",
]
FLIGHT_TYPES = {
    "AIRLINE": NVARCHAR(10), "ORIGIN_AIRPORT": NVARCHAR(10),
    "DESTINATION_AIRPORT": NVARCHAR(10), "CANCELLATION_REASON": NVARCHAR(5),
}

load_parquet("flights", DATA / "Flight/flights.parquet", FLIGHT_COLS, FLIGHT_TYPES)

flights batch 0
flights batch 1
flights batch 2
flights batch 3
flights batch 4
flights batch 5
flights batch 6
flights batch 7
flights batch 8
flights batch 9


In [6]:
from sqlalchemy import text

with engine.connect() as conn:
    total = conn.execute(
        text("SELECT COUNT_BIG(*) FROM bronze.flights")
    ).scalar_one()

print(f"Yekun sətir sayı: {total:,}")

Yekun sətir sayı: 1,000,000


In [5]:
load_parquet("weather", DATA / "api/weather_2015_all.parquet")

weather batch 0
weather batch 1
weather batch 2
weather batch 3
weather batch 4
weather batch 5
weather batch 6
weather batch 7
weather batch 8
weather batch 9
weather batch 10
weather batch 11
weather batch 12
weather batch 13
weather batch 14
weather batch 15
weather batch 16
weather batch 17
weather batch 18
weather batch 19
weather batch 20
weather batch 21
weather batch 22


In [7]:
for table in ["airlines", "airports", "flights", "weather"]:
    count = pd.read_sql(text(f"SELECT COUNT(*) AS n FROM bronze.{table}"), engine)["n"][0]
    print(table, f"{count:,}")

airlines 14
airports 322
flights 1,000,000
weather 2,225,040


## Join all dataset and upload it into the bronze schema

In [10]:
display(pd.read_sql(text(f"SELECT COUNT(*) AS row_count  FROM bronze.flights"), engine))
display(pd.read_sql(text(f"SELECT COUNT(*) AS row_count FROM bronze.airports"), engine))
display(pd.read_sql(text(f"SELECT COUNT(*) AS row_count FROM bronze.airlines"), engine))
display(pd.read_sql(text(f"SELECT COUNT(*) AS row_count FROM bronze.weather"), engine))

,row_count
0,1000000


,row_count
0,322


,row_count
0,14


,row_count
0,2225040


column names for all of the datasets in the bronze schema for joinin all of the into 1 data in bronze schema

In [11]:
for table in ["airlines", "airports", "flights", "weather"]:
    columns = pd.read_sql(
        text(f"""
            SELECT COLUMN_NAME
            FROM INFORMATION_SCHEMA.COLUMNS
            WHERE TABLE_SCHEMA = 'bronze'
              AND TABLE_NAME = '{table}'
            ORDER BY ORDINAL_POSITION
        """),
        engine
    )["COLUMN_NAME"].tolist()

    print(f"\n{table}:")
    print(columns)


airlines:
['IATA_CODE', 'AIRLINE']

airports:
['IATA_CODE', 'AIRPORT', 'CITY', 'STATE', 'COUNTRY', 'LATITUDE', 'LONGITUDE']

flights:
['MONTH', 'DAY', 'DAY_OF_WEEK', 'AIRLINE', 'ORIGIN_AIRPORT', 'DESTINATION_AIRPORT', 'DISTANCE', 'SCHEDULED_DEPARTURE', 'SCHEDULED_ARRIVAL', 'DEPARTURE_DELAY', 'ARRIVAL_DELAY', 'AIR_SYSTEM_DELAY', 'SECURITY_DELAY', 'AIRLINE_DELAY', 'LATE_AIRCRAFT_DELAY', 'WEATHER_DELAY', 'CANCELLED', 'DIVERTED', 'TAXI_OUT', 'TAXI_IN', 'AIR_TIME']

weather:
['airport_code', 'observation_time', 'Latitude', 'Longitude', 'temperature', 'precipitation', 'wind_speed', 'visibility', 'weather_condition', 'surface_pressure', 'wind_direction', 'snowfall', 'wind_gusts']


In [12]:
query = text("""
    SELECT
        f.*,

        -- Airline məlumatları
        al.AIRLINE AS AIRLINE_NAME,

        -- Başlanğıc hava limanı
        origin.AIRPORT AS ORIGIN_AIRPORT_NAME,
        origin.CITY AS ORIGIN_CITY,
        origin.STATE AS ORIGIN_STATE,
        origin.COUNTRY AS ORIGIN_COUNTRY,

        -- Təyinat hava limanı
        destination.AIRPORT AS DESTINATION_AIRPORT_NAME,
        destination.CITY AS DESTINATION_CITY,
        destination.STATE AS DESTINATION_STATE,
        destination.COUNTRY AS DESTINATION_COUNTRY

    FROM bronze.flights AS f
    LEFT JOIN bronze.airlines AS al
        ON f.AIRLINE = al.IATA_CODE
    LEFT JOIN bronze.airports AS origin
        ON f.ORIGIN_AIRPORT = origin.IATA_CODE

    LEFT JOIN bronze.airports AS destination
        ON f.DESTINATION_AIRPORT = destination.IATA_CODE
""")

In [13]:
chunks = []
total_rows = 0

for chunk in pd.read_sql(query, engine, chunksize=100_000):
    chunks.append(chunk)
    total_rows += len(chunk)
    print(f"Yüklənib: {total_rows:,} sətir")

df_flights = pd.concat(chunks, ignore_index=True)

print("\nBirləşdirmə tamamlandı.")
print("Dataset ölçüsü:", df_flights.shape)

Yüklənib: 100,000 sətir
Yüklənib: 200,000 sətir
Yüklənib: 300,000 sətir
Yüklənib: 400,000 sətir
Yüklənib: 500,000 sətir
Yüklənib: 600,000 sətir
Yüklənib: 700,000 sətir
Yüklənib: 800,000 sətir
Yüklənib: 900,000 sətir
Yüklənib: 1,000,000 sətir

Birləşdirmə tamamlandı.
Dataset ölçüsü: (1000000, 30)


In [15]:
df_flights.head(5)

,MONTH,DAY,DAY_OF_WEEK,AIRLINE,ORIGIN_AIRPORT,DESTINATION_AIRPORT,DISTANCE,SCHEDULED_DEPARTURE,SCHEDULED_ARRIVAL,DEPARTURE_DELAY,...,AIR_TIME,AIRLINE_NAME,ORIGIN_AIRPORT_NAME,ORIGIN_CITY,ORIGIN_STATE,ORIGIN_COUNTRY,DESTINATION_AIRPORT_NAME,DESTINATION_CITY,DESTINATION_STATE,DESTINATION_COUNTRY
0,10,15,4,B6,SJU,FLL,1046,1233,1512,0.0,...,145.0,JetBlue Airways,Luis Muñoz Marín International Airport,San Juan,PR,USA,Fort Lauderdale-Hollywood International Airport,Ft. Lauderdale,FL,USA
1,1,10,6,DL,ATL,MDW,591,1025,1128,-3.0,...,92.0,Delta Air Lines Inc.,Hartsfield-Jackson Atlanta International Airport,Atlanta,GA,USA,Chicago Midway International Airport,Chicago,IL,USA
2,8,21,5,WN,ATL,LAX,1947,825,1015,-1.0,...,260.0,Southwest Airlines Co.,Hartsfield-Jackson Atlanta International Airport,Atlanta,GA,USA,Los Angeles International Airport,Los Angeles,CA,USA
3,12,3,4,AA,CLT,BOS,728,940,1151,-4.0,...,85.0,American Airlines Inc.,Charlotte Douglas International Airport,Charlotte,NC,USA,Gen. Edward Lawrence Logan International Airport,Boston,MA,USA
4,7,21,2,WN,LAX,MDW,1750,1645,2240,21.0,...,202.0,Southwest Airlines Co.,Los Angeles International Airport,Los Angeles,CA,USA,Chicago Midway International Airport,Chicago,IL,USA


In [14]:
query_missing_summary = text("""
    SELECT
        COUNT(*) AS TOTAL_FLIGHTS,

        SUM(CASE
            WHEN al.IATA_CODE IS NULL THEN 1
            ELSE 0
        END) AS MISSING_AIRLINE,

        SUM(CASE
            WHEN origin.IATA_CODE IS NULL THEN 1
            ELSE 0
        END) AS MISSING_ORIGIN_AIRPORT,

        SUM(CASE
            WHEN destination.IATA_CODE IS NULL THEN 1
            ELSE 0
        END) AS MISSING_DESTINATION_AIRPORT

    FROM bronze.flights AS f

    LEFT JOIN bronze.airlines AS al
        ON f.AIRLINE = al.IATA_CODE

    LEFT JOIN bronze.airports AS origin
        ON f.ORIGIN_AIRPORT = origin.IATA_CODE

    LEFT JOIN bronze.airports AS destination
        ON f.DESTINATION_AIRPORT = destination.IATA_CODE
""")

df_missing_summary = pd.read_sql(query_missing_summary, engine)

df_missing_summary

,TOTAL_FLIGHTS,MISSING_AIRLINE,MISSING_ORIGIN_AIRPORT,MISSING_DESTINATION_AIRPORT
0,1000000,0,183,160


In [16]:
import gc

chunk_size = 100_000
cleaned_chunks = []

rows_before = len(df_flights)
rows_removed = 0

for start in range(0, rows_before, chunk_size):
    end = min(start + chunk_size, rows_before)

    # Cari hissəni götür
    chunk = df_flights.iloc[start:end]

    # Hava limanı məlumatları olmayan sətirləri sil
    clean_chunk = chunk.dropna(
        subset=[
            "ORIGIN_AIRPORT_NAME",
            "DESTINATION_AIRPORT_NAME"
        ]
    )

    rows_removed += len(chunk) - len(clean_chunk)
    cleaned_chunks.append(clean_chunk)

    print(
        f"Emal edildi: {end:,}/{rows_before:,} | "
        f"Silinib: {rows_removed:,}"
    )

# Təmizlənmiş hissələri birləşdir
df_flights_clean = pd.concat(
    cleaned_chunks,
    ignore_index=True
)

print("\nTəmizləmə tamamlandı.")
print(f"Əvvəlki sətir sayı: {rows_before:,}")
print(f"Silinən sətir sayı: {rows_removed:,}")
print(f"Qalan sətir sayı: {len(df_flights_clean):,}")

del cleaned_chunks
gc.collect()

Emal edildi: 100,000/1,000,000 | Silinib: 32
Emal edildi: 200,000/1,000,000 | Silinib: 70
Emal edildi: 300,000/1,000,000 | Silinib: 99
Emal edildi: 400,000/1,000,000 | Silinib: 141
Emal edildi: 500,000/1,000,000 | Silinib: 174
Emal edildi: 600,000/1,000,000 | Silinib: 209
Emal edildi: 700,000/1,000,000 | Silinib: 248
Emal edildi: 800,000/1,000,000 | Silinib: 287
Emal edildi: 900,000/1,000,000 | Silinib: 312
Emal edildi: 1,000,000/1,000,000 | Silinib: 343

Təmizləmə tamamlandı.
Əvvəlki sətir sayı: 1,000,000
Silinən sətir sayı: 343
Qalan sətir sayı: 999,657


0

In [17]:
df_missing_summary_clean = pd.DataFrame([{
    "TOTAL_FLIGHTS": len(df_flights_clean),

    "MISSING_AIRLINE": (
        df_flights_clean["AIRLINE_NAME"].isna().sum()
    ),

    "MISSING_ORIGIN_AIRPORT": (
        df_flights_clean["ORIGIN_AIRPORT_NAME"].isna().sum()
    ),

    "MISSING_DESTINATION_AIRPORT": (
        df_flights_clean["DESTINATION_AIRPORT_NAME"].isna().sum()
    )
}])

df_missing_summary_clean

,TOTAL_FLIGHTS,MISSING_AIRLINE,MISSING_ORIGIN_AIRPORT,MISSING_DESTINATION_AIRPORT
0,999657,0,0,0


## Creating the `Flights_total` schema in bronze for later cleaning and etc

total of columns in `Flights_total` is 40 columns all of the cleaning will be done in next steps

In [18]:
# Settings
chunk_size = 100000
target_table = "flights_total"
total_written = 0

# Load weather data from SQL Server
weather_query = text("""
    SELECT
        airport_code,
        observation_time,
        temperature AS WEATHER_TEMPERATURE,
        precipitation AS WEATHER_PRECIPITATION,
        wind_speed AS WEATHER_WIND_SPEED,
        visibility AS WEATHER_VISIBILITY,
        weather_condition AS WEATHER_CONDITION,
        surface_pressure AS WEATHER_SURFACE_PRESSURE,
        wind_direction AS WEATHER_WIND_DIRECTION,
        snowfall AS WEATHER_SNOWFALL,
        wind_gusts AS WEATHER_WIND_GUSTS
    FROM bronze.weather
""")

df_weather = pd.read_sql(weather_query, engine)

df_weather["observation_time"] = pd.to_datetime(
    df_weather["observation_time"]
)


In [19]:

# Prepare weather lookup using airport and observation hour
df_weather = df_weather.drop_duplicates(
    subset=["airport_code", "observation_time"]
)

# Process cleaned flights without copying the entire DataFrame
for start in range(0, len(df_flights_clean), chunk_size):
    end = min(start + chunk_size, len(df_flights_clean))

    # Copy only the current chunk
    chunk = df_flights_clean.iloc[start:end].copy()

    # Build scheduled flight datetime
    flight_date = pd.to_datetime({
        "year": 2015,
        "month": chunk["MONTH"],
        "day": chunk["DAY"]
    }, errors="coerce")

    departure = pd.to_numeric(
        chunk["SCHEDULED_DEPARTURE"],
        errors="coerce"
    )

    departure_minutes = (
        (departure // 100) * 60
        + (departure % 100)
    )

    chunk["FLIGHT_WEATHER_TIME"] = (
        flight_date
        + pd.to_timedelta(departure_minutes, unit="m")
    ).dt.floor("h")

    # Join weather by origin airport and scheduled departure hour
    chunk = chunk.merge(
        df_weather,
        how="left",
        left_on=["ORIGIN_AIRPORT", "FLIGHT_WEATHER_TIME"],
        right_on=["airport_code", "observation_time"],
        validate="many_to_one"
    )

    # Remove temporary join columns
    chunk.drop(
        columns=["airport_code", "observation_time"],
        inplace=True,
        errors="ignore"
    )

    # Write to SQL Server
    chunk.to_sql(
        name=target_table,
        con=engine,
        schema="bronze",
        if_exists="replace" if start == 0 else "append",
        index=False,
        chunksize=1000
    )

    total_written += len(chunk)

    print(
        f"Yazıldı: {total_written:,} / "
        f"{len(df_flights_clean):,} sətir"
    )

    del chunk, flight_date, departure, departure_minutes

print("\nBirləşdirmə tamamlandı.")
print(f"Yazılmış sətirlər: {total_written:,}")
print("Cədvəl: bronze.flights_total")


Yazıldı: 100,000 / 999,657 sətir
Yazıldı: 200,000 / 999,657 sətir
Yazıldı: 300,000 / 999,657 sətir
Yazıldı: 400,000 / 999,657 sətir
Yazıldı: 500,000 / 999,657 sətir
Yazıldı: 600,000 / 999,657 sətir
Yazıldı: 700,000 / 999,657 sətir
Yazıldı: 800,000 / 999,657 sətir
Yazıldı: 900,000 / 999,657 sətir
Yazıldı: 999,657 / 999,657 sətir

Birləşdirmə tamamlandı.
Yazılmış sətirlər: 999,657
Cədvəl: bronze.flights_total


## Making sure if all the tabel are working properly

In [24]:
print('-------Flights--------')
display(pd.read_sql(text(f"SELECT COUNT(*) AS row_count  FROM bronze.flights"), engine))
print('-------Airports--------')
display(pd.read_sql(text(f"SELECT COUNT(*) AS row_count FROM bronze.airports"), engine))
print('-------Airlines--------')
display(pd.read_sql(text(f"SELECT COUNT(*) AS row_count FROM bronze.airlines"), engine))
print('-------Weather--------')
display(pd.read_sql(text(f"SELECT COUNT(*) AS row_count FROM bronze.weather"), engine))
print('-------Flights_total--------')
display(pd.read_sql(text(f"SELECT COUNT(*) AS row_count  FROM bronze.flights_total"), engine))


-------Flights--------


,row_count
0,1000000


-------Airports--------


,row_count
0,322


-------Airlines--------


,row_count
0,14


-------Weather--------


,row_count
0,2225040


-------Flights_total--------


,row_count
0,999657


## bura yazdığımı kim görürsə böyük ehtimal o işliyəcək silver schema-ya yazmağı bunu onna görə yazıram ki yeni bir notebook yaradılsın və adı bronze clean kimi yazılsın və bronze-da yəni `flights_total`-da təmizlənmiş hər şeydən sonra bir başa ordan yəni yeni yaradılmış notebookdan bir başa ordan silver-ə yazılsın və ordan inkşaf etdirilsin   bu yazı da silsinsin. Əlavə olaraq kodlara baxarsız başa düşülmüyən nəsə olsa qarışıq qalan nəsə olsa soruşun zəhmət olmasa bəzi kodlar uzun ola bilər ama məntiqi çox sadədir izah edərəm

In [5]:
from sqlalchemy import text
import pandas as pd

query = """
SELECT TABLE_SCHEMA, TABLE_NAME
FROM INFORMATION_SCHEMA.TABLES
WHERE TABLE_SCHEMA = 'bronze'
  AND TABLE_TYPE = 'BASE TABLE'
ORDER BY TABLE_NAME;
"""

df_tables = pd.read_sql(text(query), engine)
display(df_tables)

,TABLE_SCHEMA,TABLE_NAME
0,bronze,airlines
1,bronze,airports
2,bronze,flights
3,bronze,flights_total
4,bronze,weather


In [6]:
pd.read_sql(text('''select count(*) from bronze.flights_total'''),engine)

,
0,999657
